In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
 

pd.set_option("display.max_columns", 40)

DATA_HOT_SCORE = Path("data/hotscore")
OUTPUT_DIR = Path("output/revenue")
OUTPUT_DATA_DIR = Path("data/revenue")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def load_all_hotscore_files(directory=DATA_HOT_SCORE):
    files = sorted(glob.glob(str(directory / "hotscore_*.csv")))
    
    dfs = []
    
    for f in files:
        df = pd.read_csv(f)
        
        # extract date from filename
        date_str = os.path.basename(f).split("_")[1].replace(".csv", "")
        df["date"] = pd.to_datetime(date_str, format="%Y%m%d")
        
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

df = load_all_hotscore_files()

df = df.sort_values(["symbol", "date"])
df.head(5)

,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap,date
717500,A,0.943797,0.843672,138.450,19.518301,1.793232,2151527.0,0.985112,0.923077,0.957816,3.912629e+10,2026-05-28
717549,A,0.938902,0.827103,135.380,16.868100,2.574832,2151527.0,0.976636,0.927570,0.948598,3.825870e+10,2026-05-28
0,AA,0.794401,0.520833,41.845,6.747450,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
50,AA,0.794401,0.520833,41.845,6.747450,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
100,AA,0.773989,0.557951,41.560,6.020410,1.039631,6727448.0,0.876011,0.749326,0.746631,1.076255e+10,2026-01-17


In [4]:
price_stats = df.groupby("symbol").agg(
    min_price=("regularMarketPrice", "min"),
    max_price=("regularMarketPrice", "max"),
    mean_price=("regularMarketPrice", "mean")
)

price_stats["range_pct"] = (
    (price_stats["max_price"] - price_stats["min_price"]) 
    / price_stats["mean_price"]
)

In [5]:
df["direction"] = np.sign(df["regularMarketChangePercent"])

direction_changes = (
    df.groupby("symbol")["direction"]
    .apply(lambda x: (x.diff() != 0).sum())
    .to_frame(name="direction_changes")
)

In [6]:
metrics = df.groupby("symbol").agg(
    avg_volatility=("VolatilityScore", "mean"),
    max_volatility=("VolatilityScore", "max"),
    avg_momentum=("MomentumScore", "mean"),
    avg_volume=("VolumeScore", "mean"),
    avg_volume_spike=("VolumeSpike", "mean"),
    avg_change_pct=("regularMarketChangePercent", "mean")
)

In [7]:
final = price_stats.join(direction_changes).join(metrics)

# Normalize direction_changes
final["dir_change_norm"] = final["direction_changes"].rank(pct=True)

# Swing Score formula
final["swing_score"] = (
    final["range_pct"] * 0.35 +
    final["dir_change_norm"] * 0.30 +
    final["avg_volatility"] * 0.20 +
    final["avg_volume"] * 0.15
)

final = final.sort_values("swing_score", ascending=False)

In [8]:
swing_candidates = final[
    (final["avg_volatility"] > 0.6) &
    (final["avg_volume"] > 0.5) &
    (final["range_pct"] > 0.25)
].sort_values("swing_score", ascending=False)

top_swings = swing_candidates.head(20)

top_swings.head(10)

,min_price,max_price,mean_price,range_pct,direction_changes,avg_volatility,max_volatility,avg_momentum,avg_volume,avg_volume_spike,avg_change_pct,dir_change_norm,swing_score
symbol,,,,,,,,,,,,,
SNDK,202.4600,1562.3400,288.096882,4.720218,137,0.964159,1.000000,0.835794,0.749435,0.829592,7.274895,0.993596,2.255402
AAOI,29.3500,223.1000,41.888082,4.625421,3,0.772332,1.000000,0.938490,0.836719,1.212836,11.303639,0.854680,2.155275
NOW,88.4300,807.4500,165.623885,4.341282,37,0.817853,0.992126,0.556966,0.781467,0.726397,2.105953,0.959606,2.088121
MU,234.8050,928.4100,294.193758,2.357647,259,0.925227,1.000000,0.751820,0.801984,0.983750,4.957463,1.000000,1.430519
ARM,107.5800,410.7101,125.751514,2.410548,48,0.821064,0.992941,0.675212,0.849016,1.141438,4.545593,0.966749,1.425282
MRVL,86.7600,316.4300,97.852869,2.347095,25,0.851711,1.000000,0.791104,0.719886,1.258382,4.654758,0.937192,1.380966
DELL,118.1600,466.4900,138.042010,2.523362,1,0.903201,1.000000,0.839582,0.907422,1.900456,6.143690,0.417241,1.325103
BE,80.7100,315.1100,108.840836,2.153603,25,0.904179,0.997647,0.905011,0.671529,0.742974,8.310092,0.937192,1.316484
INTC,35.7099,124.9200,42.504120,2.098858,149,0.702141,0.955224,0.813089,0.715343,0.790486,6.253726,0.994828,1.280778


In [26]:
def score_company_health(metrics_df):
    """
    Score companies based on financial health metrics
    Scoring criteria:
    - Lower P/E ratio (better valuation)
    - Higher profit margins (profitability)
    - Higher ROE/ROA (efficiency)
    - Lower debt/equity (financial stability)
    - Higher current ratio (liquidity)
    - Positive revenue growth
    """
    
    df = metrics_df.copy()
    
    # Convert columns to numeric
    for col in ['trailingPE', 'forwardPE', 'profitMargins', 'operatingMargins', 
                'returnOnEquity', 'returnOnAssets', 'debtToEquity', 'currentRatio', 
                'quickRatio', 'revenueGrowth', 'dividendYield']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Initialize scores
    df['health_score'] = 0.0
    
    # 1. P/E Ratio Scoring (lower is better, max 25 points)
    # Normalize P/E: 15 is ideal (score 25), 5 and 50 both score 5
    if 'trailingPE' in df.columns:
        df['pe_score'] = df['trailingPE'].apply(
            lambda x: max(0, 25 - abs(x - 15) * 1) if pd.notna(x) and x > 0 else 0
        )
        df['health_score'] += df['pe_score']
    
    # 2. Profit Margins Scoring (higher is better, max 20 points)
    if 'profitMargins' in df.columns:
        df['profit_score'] = df['profitMargins'].apply(
            lambda x: min(20, max(0, x * 100)) if pd.notna(x) else 0
        )
        df['health_score'] += df['profit_score']
    
    # 3. Operating Margins Scoring (higher is better, max 15 points)
    if 'operatingMargins' in df.columns:
        df['op_margin_score'] = df['operatingMargins'].apply(
            lambda x: min(15, max(0, x * 100)) if pd.notna(x) else 0
        )
        df['health_score'] += df['op_margin_score']
    
    # 4. ROE Scoring (higher is better, max 15 points)
    if 'returnOnEquity' in df.columns:
        df['roe_score'] = df['returnOnEquity'].apply(
            lambda x: min(15, max(0, x * 100)) if pd.notna(x) else 0
        )
        df['health_score'] += df['roe_score']
    
    # 5. Debt to Equity Scoring (lower is better, max 10 points)
    if 'debtToEquity' in df.columns:
        df['debt_score'] = df['debtToEquity'].apply(
            lambda x: max(0, 10 - (x if pd.notna(x) else 0)) if pd.notna(x) else 5
        )
        df['health_score'] += df['debt_score']
    
    # 6. Current Ratio Scoring (higher is better, ideal 1.5-2.0, max 10 points)
    if 'currentRatio' in df.columns:
        df['liquidity_score'] = df['currentRatio'].apply(
            lambda x: max(0, 10 - abs(x - 1.5) * 5) if pd.notna(x) else 0
        )
        df['health_score'] += df['liquidity_score']
    
    # 7. Revenue Growth Scoring (higher is better, max 15 points)
    if 'revenueGrowth' in df.columns:
        df['growth_score'] = df['revenueGrowth'].apply(
            lambda x: min(15, max(-5, x * 100)) if pd.notna(x) else 0
        )
        df['health_score'] += df['growth_score']
    
    return df

In [39]:
import yfinance as yf

def get_ticker_metrics(symbol):
    """Fetch key metrics from a ticker for scoring"""
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        
        metrics = {
            'symbol': symbol,
            'marketCap': info.get('marketCap', None),
            'trailingPE': info.get('trailingPE', None),
            'forwardPE': info.get('forwardPE', None),
            'dividendYield': info.get('dividendYield', None),
            'operatingMargins': info.get('operatingMargins', None),
            'debtToEquity': info.get('debtToEquity', None),
            'returnOnAssets': info.get('returnOnAssets', None),
            'returnOnEquity': info.get('returnOnEquity', None),
            'profitMargins': info.get('profitMargins', None),
            'currentRatio': info.get('currentRatio', None),
            'quickRatio': info.get('quickRatio', None),
            '52WeekChange': info.get('52WeekChange', None),
            'revenueGrowth': info.get('revenueGrowth', None),
            'currentPrice': info.get('currentPrice', None),
        }
        
        return metrics
    except Exception as e:
        print(f"Error fetching metrics for {symbol}: {e}")
        return None


In [ ]:
# ============================================
# COMPLETE ANALYSIS: HEALTH SCORES + SWING METRICS
# ============================================

print(f"\n{'='*80}")
print(f"COMPLETE ANALYSIS: SCORING 500 COMPANIES & FINDING TOP 50")
print(f"{'='*80}\n")

# Step 1: Get metrics from 500 swing candidates
top_symbols = swing_candidates.head(500).index.tolist()
print(f"Fetching metrics for {len(top_symbols)} companies...")

all_metrics = []
for i, symbol in enumerate(top_symbols):
    metrics = get_ticker_metrics(symbol)
    if metrics:
        all_metrics.append(metrics)
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(top_symbols)}...")

metrics_df = pd.DataFrame(all_metrics)
print(f"✓ Fetched metrics for {len(metrics_df)} companies\n")

# Step 2: Score companies by financial health
print("Scoring companies by financial health...")
scored_df = score_company_health(metrics_df)

# Step 3: Get top 50 healthiest
top_50_healthy = scored_df.nlargest(50, 'health_score')
print(f"✓ Identified top 50 healthiest companies\n")

# Step 4: Merge with swing metrics
print("Merging with swing trading metrics...")
top_50_with_swing = top_50_healthy.copy()

# Add swing metrics
for idx, row in top_50_with_swing.iterrows():
    symbol = row['symbol']
    if symbol in final.index:
        swing_data = final.loc[symbol]
        top_50_with_swing.at[idx, 'swing_score'] = swing_data.get('swing_score', 0)
        top_50_with_swing.at[idx, 'avg_volatility'] = swing_data.get('avg_volatility', 0)
        top_50_with_swing.at[idx, 'avg_volume_spike'] = swing_data.get('avg_volume_spike', 0)
        top_50_with_swing.at[idx, 'range_pct'] = swing_data.get('range_pct', 0)
    else:
        top_50_with_swing.at[idx, 'swing_score'] = 0
        top_50_with_swing.at[idx, 'avg_volatility'] = 0
        top_50_with_swing.at[idx, 'avg_volume_spike'] = 0
        top_50_with_swing.at[idx, 'range_pct'] = 0

# Calculate composite score
max_health = top_50_with_swing['health_score'].max() if top_50_with_swing['health_score'].max() > 0 else 1
max_swing = top_50_with_swing['swing_score'].max() if top_50_with_swing['swing_score'].max() > 0 else 1

top_50_with_swing['composite_score'] = (
    (top_50_with_swing['health_score'] / max_health) * 50 +
    (top_50_with_swing['swing_score'] / max_swing) * 50
)

top_50_with_swing = top_50_with_swing.sort_values('composite_score', ascending=False)
print(f"✓ Composite scoring complete\n")



COMPLETE ANALYSIS: SCORING 500 COMPANIES & FINDING TOP 50

Fetching metrics for 500 companies...
  Processed 100/500...
  Processed 200/500...
  Processed 300/500...
  Processed 400/500...
  Processed 500/500...
✓ Fetched metrics for 500 companies

Scoring companies by financial health...
✓ Identified top 50 healthiest companies

Merging with swing trading metrics...
✓ Composite scoring complete



: 

In [36]:
# ============================================
# VISUALIZE TOP 50 HEALTHY COMPANIES
# ============================================

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"\n{'='*80}")
print("GENERATING VISUALIZATIONS")
print(f"{'='*80}\n")

# Prepare data for visualizations
top_50_chart = top_50_with_swing.head(50).copy()

# Chart 1: Health Score vs Composite Score (Top 20)
top_20 = top_50_chart.head(20)

fig1 = px.scatter(
    top_20,
    x='health_score',
    y='composite_score',
    size='avg_volatility',
    color='trailingPE',
    text='symbol',
    title='Top 20 Companies: Health vs Composite Score<br><sub>Size=Volatility | Color=P/E Ratio</sub>',
    hover_data={
        'profitMargins': ':.1%',
        'returnOnEquity': ':.1%',
        'debtToEquity': ':.2f',
        'health_score': ':.1f',
        'composite_score': ':.1f'
    },
    color_continuous_scale='RdYlGn_r'
)

fig1.update_traces(textposition='top center', textfont_size=11)
fig1.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font_color='white',
    height=600,
    xaxis_title='Health Score',
    yaxis_title='Composite Score (Health + Swing Opportunity)'
)

chart1_path = os.path.join(OUTPUT_DIR, "top_50_health_vs_composite.html")
fig1.write_html(chart1_path, include_plotlyjs='cdn')
print("✓ Saved: top_50_health_vs_composite.html")

# Chart 2: Ranking Bar Chart (Top 20)
top_20_sorted = top_20.sort_values('composite_score', ascending=True)

fig2 = px.bar(
    top_20_sorted,
    y='symbol',
    x='composite_score',
    orientation='h',
    color='health_score',
    text='composite_score',
    color_continuous_scale='Viridis',
    title='Top 20 Investment Opportunities Ranked<br><sub>Color = Health Score</sub>',
    hover_data={
        'trailingPE': ':.2f',
        'profitMargins': ':.1%',
        'returnOnEquity': ':.1%',
        'avg_volatility': ':.3f'
    }
)

fig2.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig2.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font_color='white',
    height=600,
    xaxis_title='Composite Score'
)

chart2_path = os.path.join(OUTPUT_DIR, "top_20_ranking.html")
fig2.write_html(chart2_path, include_plotlyjs='cdn') 

# Chart 3: Financial Metrics Heatmap (Top 20)
heatmap_data = top_20[['symbol', 'profitMargins', 'returnOnEquity', 'health_score', 'avg_volatility']].copy()
heatmap_data['profitMargins'] = heatmap_data['profitMargins'] * 100
heatmap_data['returnOnEquity'] = heatmap_data['returnOnEquity'] * 100
heatmap_data = heatmap_data.set_index('symbol')
heatmap_data = heatmap_data.round(1)

fig3 = go.Figure(
    data=go.Heatmap(
        z=heatmap_data.T.values,
        x=heatmap_data.index,
        y=['Profit Margin %', 'ROE %', 'Health Score', 'Volatility'],
        colorscale='RdYlGn',
        text=heatmap_data.T.values.round(1),
        texttemplate='%{text:.1f}',
        textfont={"size": 8}
    )
)

fig3.update_layout(
    title='Top 20 Companies: Financial Metrics Heatmap',
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font_color='white',
    height=500
)

chart3_path = os.path.join(OUTPUT_DIR, "top_20_metrics_heatmap.html")
fig3.write_html(chart3_path, include_plotlyjs='cdn') 

# Chart 4: P/E vs Profit Margin (Risk-Return)
fig4 = px.scatter(
    top_50_chart.head(30),
    x='trailingPE',
    y='profitMargins',
    size='composite_score',
    color='health_score',
    text='symbol',
    title='Risk-Return Profile: P/E vs Profit Margin<br><sub>Size=Composite Score | Color=Health</sub>',
    color_continuous_scale='Viridis'
)

fig4.update_traces(textposition='top center', textfont_size=9)
fig4.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font_color='white',
    height=600,
    xaxis_title='P/E Ratio (Lower = Cheaper)',
    yaxis_title='Profit Margin % (Higher = Better)'
)

chart4_path = os.path.join(OUTPUT_DIR, "top_20_risk_return.html")
fig4.write_html(chart4_path, include_plotlyjs='cdn') 

# Chart 5: Volatility vs Composite Score Distribution
fig5 = px.scatter(
    top_50_chart,
    x='avg_volatility',
    y='composite_score',
    size='health_score',
    color='trailingPE',
    text='symbol',
    title='All 50 Companies: Volatility vs Composite Score<br><sub>Size=Health Score | Color=P/E</sub>',
    color_continuous_scale='RdYlGn_r',
    opacity=0.7
)

fig5.update_traces(textposition='middle center', textfont_size=8, marker_line_width=0.5)
fig5.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font_color='white',
    height=700,
    xaxis_title='Average Volatility',
    yaxis_title='Composite Score'
)

chart5_path = os.path.join(OUTPUT_DIR, "top_20_volatility_distribution.html")
fig5.write_html(chart5_path, include_plotlyjs='cdn') 

print("\n✅ All visualizations generated and saved to output/revenue/")



GENERATING VISUALIZATIONS

✓ Saved: top_50_health_vs_composite.html

✅ All visualizations generated and saved to output/revenue/


In [37]:
# Display results
print("="*140)
print("TOP 50 HEALTHIEST COMPANIES WITH SWING OPPORTUNITY SCORES")
print("="*140 + "\n")

display_cols = ['symbol', 'health_score', 'composite_score', 'trailingPE', 'profitMargins',
                'returnOnEquity', 'debtToEquity', 'swing_score', 'avg_volatility']

results_display = top_50_with_swing[display_cols].head(50).copy()
results_display['health_score'] = results_display['health_score'].round(1)
results_display['composite_score'] = results_display['composite_score'].round(1)
results_display['trailingPE'] = results_display['trailingPE'].round(2)
results_display['profitMargins'] = (results_display['profitMargins'] * 100).round(1)
results_display['returnOnEquity'] = (results_display['returnOnEquity'] * 100).round(1)
results_display['debtToEquity'] = results_display['debtToEquity'].round(2)
results_display['swing_score'] = results_display['swing_score'].round(3)
results_display['avg_volatility'] = results_display['avg_volatility'].round(3)

print(results_display.head(10).to_string(index=False))

TOP 50 HEALTHIEST COMPANIES WITH SWING OPPORTUNITY SCORES

symbol  health_score  composite_score  trailingPE  profitMargins  returnOnEquity  debtToEquity  swing_score  avg_volatility
  DUOL          95.3             96.8       12.49           38.4            37.0          6.60        0.774           0.948
  PAAS          91.6             94.1       16.74           31.6            20.8         11.49        0.761           0.790
  PATH          89.6             92.3       19.45           19.6            18.2          3.78        0.749           0.657
  FSLR          93.5             92.1       20.32           30.7            18.4          5.94        0.714           0.949
   NVO          91.7             91.0       10.27           37.2            71.4         72.09        0.710           0.733
  BKNG          80.7             90.8       22.07           22.2             NaN           NaN        0.796           1.000
   NEM          94.3             89.9       14.05           33.9         

In [38]:
# ============================================
# TOP 10 INVESTMENT OPPORTUNITIES BAR CHART
# ============================================

print(f"\n{'='*80}")
print("CREATING TOP 10 BAR CHARTS")
print(f"{'='*80}\n")

# Get top 10 from results_display
top_10_data = results_display.head(10).copy()

# Chart 1: Composite Score Ranking (Top 10)
fig_top10 = px.bar(
    top_10_data.sort_values('composite_score', ascending=True),
    x='composite_score',
    y='symbol',
    orientation='h',
    color='health_score',
    text='composite_score',
    title='🎯 TOP 10 INVESTMENT OPPORTUNITIES<br><sub>Ranked by Composite Score (Health + Swing)</sub>',
    color_continuous_scale='Viridis',
    hover_data={
        'health_score': ':.1f',
        'trailingPE': ':.2f',
        'profitMargins': ':.1f',
        'returnOnEquity': ':.1f',
        'swing_score': ':.3f',
        'avg_volatility': ':.3f'
    }
)

fig_top10.update_traces(texttemplate='%{text:.1f}', textposition='outside', textfont_size=12)
fig_top10.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font=dict(color='white', size=12),
    xaxis_title='Composite Score',
    yaxis_title='Symbol',
    height=600,
    showlegend=True,
    coloraxis_colorbar=dict(title='Health Score')
)

chart_top10_path = os.path.join(OUTPUT_DIR, "top_10_opportunities.html")
fig_top10.write_html(chart_top10_path, include_plotlyjs='cdn')

# Chart 2: Multi-metric comparison (Top 10)
from plotly.subplots import make_subplots

fig_multi = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Composite Score', 'Health Score', 'P/E Ratio', 'Volatility'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

top_10_sorted = top_10_data.sort_values('composite_score', ascending=True)

# Composite Score
fig_multi.add_trace(
    go.Bar(
        x=top_10_sorted['composite_score'],
        y=top_10_sorted['symbol'],
        orientation='h',
        marker_color='#FF6B6B',
        name='Composite',
        text=top_10_sorted['composite_score'].round(1),
        textposition='outside'
    ),
    row=1, col=1
)

# Health Score
fig_multi.add_trace(
    go.Bar(
        x=top_10_sorted['health_score'],
        y=top_10_sorted['symbol'],
        orientation='h',
        marker_color='#4ECDC4',
        name='Health',
        text=top_10_sorted['health_score'].round(1),
        textposition='outside'
    ),
    row=1, col=2
)

# P/E Ratio
fig_multi.add_trace(
    go.Bar(
        x=top_10_sorted['trailingPE'],
        y=top_10_sorted['symbol'],
        orientation='h',
        marker_color='#45B7D1',
        name='P/E',
        text=top_10_sorted['trailingPE'].round(1),
        textposition='outside'
    ),
    row=2, col=1
)

# Volatility
fig_multi.add_trace(
    go.Bar(
        x=top_10_sorted['avg_volatility'],
        y=top_10_sorted['symbol'],
        orientation='h',
        marker_color='#FFA07A',
        name='Volatility',
        text=top_10_sorted['avg_volatility'].round(3),
        textposition='outside'
    ),
    row=2, col=2
)

fig_multi.update_xaxes(title_text='Score', row=1, col=1)
fig_multi.update_xaxes(title_text='Score', row=1, col=2)
fig_multi.update_xaxes(title_text='Ratio', row=2, col=1)
fig_multi.update_xaxes(title_text='Score', row=2, col=2)

fig_multi.update_layout(
    title_text='Top 10 Companies: Multi-Metric Comparison',
    height=700,
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font=dict(color='white', size=11),
    showlegend=False,
    margin=dict(l=80, r=100, t=60, b=40)
)

chart_multi_path = os.path.join(OUTPUT_DIR, "top_10_multi_metric.html")
fig_multi.write_html(chart_multi_path, include_plotlyjs='cdn')

# Chart 3: Profit Margin & Volatility Scatter (Top 10)
fig_scatter_top10 = px.scatter(
    top_10_data,
    x='profitMargins',
    y='avg_volatility',
    size='composite_score',
    color='trailingPE',
    text='symbol',
    title='Top 10: Profit Margin vs Volatility<br><sub>Size=Composite Score | Color=P/E Ratio</sub>',
    color_continuous_scale='RdYlGn_r'
)

fig_scatter_top10.update_traces(textposition='top center', textfont_size=13, marker_line_width=2)
fig_scatter_top10.update_layout(
    plot_bgcolor='#0b0f1a',
    paper_bgcolor='#0b0f1a',
    font=dict(color='white', size=12),
    height=600,
    xaxis_title='Profit Margin %',
    yaxis_title='Avg Volatility',
    showlegend=True
)

chart_scatter_path = os.path.join(OUTPUT_DIR, "top_10_scatter.html")
fig_scatter_top10.write_html(chart_scatter_path, include_plotlyjs='cdn')



CREATING TOP 10 BAR CHARTS

